In [ ]:
# start coding here
# start coding here
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from tqdm.auto import tqdm
import torch

In [ ]:
# load

scores_df = pd.read_parquet(snakemake.input.cw_transcriptome_term_scores)
scores_df.head()

In [ ]:
gsva_results = pd.read_parquet(snakemake.input.gsva_results).set_index("Unnamed: 0")
gsva_library = gsva_results.pop("library")
gsva_results.head()

In [ ]:
assert (gsva_results.columns == scores_df.columns).all()

# index does not match everywhere because we eliminated the GO terms (GO:123456) in the previous step
assert (gsva_results.index[0] == scores_df.index[0])

In [ ]:
# Correlate on a per-row basis
row_correlations = np.array([pearsonr(scores_df.iloc[i, :], gsva_results.iloc[i, :]) for i in tqdm(range(scores_df.shape[0]), total=scores_df.shape[0])])

# Correlate on a per-column basis
column_correlations = np.array([pearsonr(scores_df.iloc[:, i], gsva_results.iloc[:, i]) for i in tqdm(range(scores_df.shape[1]), total=scores_df.shape[1])])

In [ ]:
gene_corr_df = pd.DataFrame(column_correlations, index=gsva_results.columns, columns=["rho", "p"])
gene_corr_df["type"] = "gene"
term_corr_df = pd.DataFrame(row_correlations, index=gsva_results.index, columns=["rho", "p"])
term_corr_df["type"] = "term"

pd.concat([gene_corr_df, term_corr_df]).to_csv(snakemake.output.gsva_correlation_results)

gene_corrs = gene_corr_df["rho"]
term_corrs = term_corr_df["rho"]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(3, 3))

sns.ecdfplot(term_corrs, ax=ax)
ax.axhline(0.5)
ax.axvline(0.0)
ax.set_title("Distribution of term-level correlations")
plt.tight_layout()
fig.savefig(snakemake.output.term_level_correlations)

In [ ]:
# get the top hit
term_corrs.sort_values(ascending=False)

In [ ]:
#  scatterplot of one term (cherry picked)
fig, ax = plt.subplots(figsize=(3, 3))

term = snakemake.params.selected_top_term
sns.scatterplot(x=gsva_results.loc[term], y=scores_df.loc[term], s=1)
ax.set_xlabel("GSVA")
ax.set_ylabel("CellWhisperer score")
ax.set_title(f"\"{term}\"")
ax.text(6, 1, f"ρ={term_corrs.loc[term]}")
plt.tight_layout()
fig.savefig(snakemake.output.top_term_correlation)

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.5))
term_corrs.groupby(gsva_library).mean().sort_values(ascending=True).plot.barh(ax=ax)
ax.set_xlabel('correlation for 50k samples', ha="right")
plt.tight_layout()
fig.savefig(snakemake.output.library_correlations)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(3, 3))

sns.ecdfplot(term_corrs[gsva_library == "OMIM_Expanded"], ax=ax)
ax.axhline(0.5)
ax.axvline(0.0)
ax.set_title("Correlation coefficients for 'OMIM_Expanded'")
plt.tight_layout()
fig.savefig(snakemake.output.omim_correlations)

In [ ]:
plot_df = []

for i, term in enumerate(gsva_results.index):
    pos = gsva_results.iloc[i][scores_df.iloc[i] > 0]
    neg = gsva_results.iloc[i][scores_df.iloc[i] <= 0]
    plot_df.append(
        pd.DataFrame({
            "term": term,
            "type": "positive CW score",
            "ids": pos.index,
            "gsva_score": pos.values
        })
    )
    plot_df.append(
        pd.DataFrame({
            "term": term,
            "type": "negative CW score",
            "ids": neg.index,
            "gsva_score": neg.values
        })
    )
plot_df = pd.concat(plot_df)

In [ ]:
plot_df["term"] = pd.Categorical(plot_df.term)

In [ ]:
# TODO select some interesting terms

subdf = plot_df[plot_df.term.isin(["Pluripotent Stem Cells", "Lung V2 (HLCA)-ann Level 2-Smooth Muscle"])].copy()
subdf["term"] = subdf["term"].astype(str)  # plotting this is ass slow

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

sns.violinplot(data=subdf.iloc[::-1], x="term", y="gsva_score", hue="type", ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), ha="right", rotation=10)
plt.tight_layout()

fig.savefig(snakemake.output.cw_binarized_gsva_scores)